In [ ]:
import os
from pathlib import Path
import pathlib
from dotenv import load_dotenv
from ipumspy import IpumsApiClient, MicrodataExtract, readers, ddi



In [ ]:
data_dir = pathlib.Path('data')
parquet_dir = data_dir / "cps.parquet"
parquet_dir.parent.mkdir(exist_ok=True)


In [ ]:
if not data_dir.glob("*.xml"):
    raise FileNotFoundError(f"No .xml codebook found in {data_dir}/")

# Get the DDI
ddi_file = list(data_dir.glob("*.xml"))[0]
ddi = readers.read_ipums_ddi(ddi_file)

# Get the data
ipums_df = readers.read_microdata(ddi, data_dir / ddi.file_description.filename)

In [ ]:
ipums_df.columns

In [ ]:
col_names = ipums_df.columns.to_list()
for col in col_names:
    col_info = ddi.get_variable_info(col)
    print(f"{col_info.label} ({col}):")
    print(f"{col_info.description}")
    print(f"{col_info.codes}")
    print("="*50)
    # print(f"{col}: {col_info.label} --> \n {col_info.description} {col_info.codes}\n ({col_info.name})")

In [ ]:
print(f"Rows x cols : {ipums_df.shape}")
print(f"Columns     : {list(ipums_df.columns)}")
print(f"Years        : {sorted(ipums_df['YEAR'].unique().tolist())}")
if "ASECWT" not in ipums_df.columns:
    raise SystemExit(
        "FATAL: ASECWT (person weight) is missing. Every rate would be "
        "unweighted and wrong. Re-run extract.py with ASECWT in the list."
    )

ipums_df.to_parquet(parquet_dir)
print(f"\nWrote {parquet_dir}  ({parquet_dir.stat().st_size/1e6:.1f} MB)")